In [ ]:
###Tictactoe reference code : https://codelearn.io/sharing/day-ai-danh-tictactoe-voi-deep-learning
##wandb link : https://wandb.ai/kradeero-ohio-university/experiments/runs/3cmbdocf

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class BaseModel: 
    def __init__(self, discount_factor, epsilon, e_min, e_max):
        self.discount_factor = discount_factor
        self.epsilon = epsilon
        self.e_min = e_min
        self.e_max = e_max
        pass

class Tictactoe_v0:
    def __init__(self):
        self.board = [0] * 9
        self.wining_position = [[0, 1, 2], [3, 4, 5], [6, 7, 8],
                                [0, 3, 6], [1, 4, 7], [2, 5, 8],
                                [0, 4, 8], [6, 4, 2]]
        self.current_turn = 1
        self.player_mark = 1

    def reset(self, is_human_first):
        self.board = [0] * 9
        self.current_turn = 1
        self.player_mark = 1 if is_human_first else -1

        # uncomment the below
        # print("\n--- New Episode ---")
        # self.render() 

        # # *Then let O play first if X is not first*
        # if not is_human_first:
        #     reward, done = self.env_act()
        #     self.render()  # Render after O's move
        if not is_human_first:
            self.env_act()
            #self.render()
        return self.board.copy()

    def check_win(self):
        for pst in self.wining_position:
            if str(self.board[pst[0]]) + str(self.board[pst[1]]) + str(self.board[pst[2]]) in ['111', '-1-1-1']:
                if self.current_turn == self.player_mark:
                    return 1, True
                return -1, True
        if 0 not in self.board:
            return 0, True
        return 0, False

    def env_act(self):
        action = random.choice([i for i in range(len(self.board)) if self.board[i] == 0])
        for pst in self.wining_position:
            com = str(self.board[pst[0]]) + str(self.board[pst[1]]) + str(self.board[pst[2]])
            if com.replace('0', '') == str(self.current_turn) * 2:
                if self.board[pst[0]] == 0:
                    action = pst[0]
                elif self.board[pst[1]] == 0:
                    action = pst[1]
                else:
                    action = pst[2]
        if self.board[action] != 0:
            raise Exception('Invalid action')
        self.board[action] = self.current_turn
        reward, done = self.check_win()
        self.current_turn = self.current_turn * -1
        return reward, done

    def step(self, action):
        if self.board[action] != 0:
            raise Exception('Invalid action')
        self.board[action] = self.current_turn
        # uncomment the below
        # self.render()
        reward, done = self.check_win()
        self.current_turn = self.current_turn * -1
        if not done:
            reward, done = self.env_act()
            # uncomment the below
            # self.render()  # Render after O move
        return self.board.copy(), reward, done, None

    
    def render(self):
        """Visualize the board with X/O positions"""
        symbols = {1: 'X', -1: 'O', 0: ' '}
        print("\nCurrent Board:")
        for i in range(3):
            print(f" {symbols[self.board[i*3]]} | {symbols[self.board[i*3+1]]} | {symbols[self.board[i*3+2]]} ")
            if i < 2: print("-----------")
        print()

class EpsilonGreedy:
    def __init__(self, epsilon):
        self.epsilon = epsilon

    def perform(self, q_value, action_space: list = None):
        prob = np.random.sample()  # get probability of taking random action
        if prob <= self.epsilon:  # take random action
            if action_space is None:  # all action are available
                return np.random.randint(len(q_value))
            return np.random.choice(action_space)
        else:  # take greedy action
            if action_space is None:
                return np.argmax(q_value)
            return max([[q_value[a], a] for a in action_space], key=lambda x: x[0])[1]

    def decay(self, decay_value, lower_bound):
        """
        Adjust the epsilon value by the formula: epsilon = max(decayValue * epsilon, lowerBound).
        :param decay_value: Value ratio adjustment (0, 1).
        :param lower_bound: Minimum epsilon value.
        :return: None
        """
        self.epsilon = max(self.epsilon * decay_value, lower_bound)


class ExperienceReplay:
    def __init__(self, e_max: int):
        if e_max <= 0:
            raise ValueError('Invalid value for memory size')
        self.e_max = e_max
        self.memory = list()
        self.index = 0

    def add_experience(self, sample: list):
        if len(sample) != 5:
            raise Exception('Invalid sample')
        if len(self.memory) < self.e_max:
            self.memory.append(sample)
        else:
            self.memory[self.index] = sample
        self.index = (self.index + 1) % self.e_max

    def sample_experience(self, sample_size: int, cer_mode: bool):
        samples = random.sample(self.memory, sample_size)
        if cer_mode:
            samples[-1] = self.memory[self.index - 1]
        # state_samples, action_samples, reward_samples, next_state_samples, done_samples
        s_batch, a_batch, r_batch, ns_batch, done_batch = map(np.array, zip(*samples))
        return s_batch, a_batch, r_batch, ns_batch, done_batch

    def get_size(self):
        return len(self.memory)
    


class DQN(BaseModel):
    def __init__(self, discount_factor: float, epsilon: float, e_min: int, e_max: int):
        super().__init__(discount_factor, epsilon, e_min, e_max)
        self.gamma = discount_factor
        self.epsilon_greedy = EpsilonGreedy(1.0)
        self.e_min = e_min
        self.exp_replay = ExperienceReplay(e_max)
        self.training_network = nn.Sequential(
            nn.Linear(9, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 9)
        )
        self.target_network = nn.Sequential(
            nn.Linear(9, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 9)
        )
        # self.optimizer = optim.RMSprop(self.training_network.parameters(), lr=0.00025)
        self.optimizer = optim.Adam(self.training_network.parameters(), lr=0.0001)
        self.criterion = nn.MSELoss()
        self.cache = list()


    def observe(self, state, action_space: list = None):
        state_tensor = torch.Tensor(np.array([state])).float() # Convert to PyTorch Tensor
        q_value = self.training_network(state_tensor).detach().numpy().ravel() # Forward pass and convert back to numpy
        if action_space is not None:
            return max([[q_value[a], a] for a in action_space], key=lambda x: x[0])[1]
        return np.argmax(q_value)

    def observe_on_training(self, state, action_space: list = None) -> int:
        state_tensor = torch.Tensor(np.array([state])).float() # Convert to PyTorch Tensor
        q_value = self.training_network(state_tensor).detach().numpy().ravel() # Forward pass and convert back to numpy
        action = self.epsilon_greedy.perform(q_value, action_space)
        self.cache.extend([state, action])
        return action

    def take_reward(self, reward, next_state, done):
        norm_reward = np.clip(reward, -1, 1)
        self.cache.extend([norm_reward, next_state, done])
        self.exp_replay.add_experience(self.cache.copy())
        self.cache.clear()

    def train_network(self, sample_size: int, batch_size: int, epochs: int, verbose: int = 2, cer_mode: bool = False):
        if self.exp_replay.get_size() >= self.e_min:
            s_batch, a_batch, r_batch, ns_batch, done_batch = self.exp_replay.sample_experience(sample_size, cer_mode)

            # Convert batches to PyTorch tensors
            state_batch = torch.Tensor(s_batch).float()
            action_batch = torch.LongTensor(a_batch) # Actions are usually Long tensors for indexing
            reward_batch = torch.Tensor(r_batch).float()
            next_state_batch = torch.Tensor(ns_batch).float()
            done_batch = torch.Tensor(done_batch).float() # or BoolTensor if you prefer

            states, q_values = self.replay(state_batch, action_batch, reward_batch, next_state_batch, done_batch)
            self.optimizer.zero_grad()  # Use the optimizer from __init__
            predictions = self.training_network(states)
            loss = self.criterion(predictions, q_values)
            loss.backward()
            self.optimizer.step()
            return loss.item()
        return None

            # # # Training loop in PyTorch (manual fit)
            # # optimizer = optim.RMSprop(self.training_network.parameters(), lr=0.00025) # Define optimizer here or in init

            # optimizer = optim.Adam(self.training_network.parameters(), lr=0.00025)
            # criterion = nn.MSELoss() # Define loss function here or in init

            # optimizer.zero_grad() # Zero gradients before each batch
            # predictions = self.training_network(states) # Forward pass
            # loss = criterion(predictions, q_values) # Calculate loss
            # loss.backward() # Backpropagation
            # optimizer.step() # Update weights

            # return loss.item() # Return loss value

    # def replay(self, states, actions, rewards, next_states, terminals):
    #     q_values_target = self.target_network(next_states).detach() # Detach from computation graph for target network
    #     max_next_q_values = q_values_target.max(dim=1)[0] # Get max Q-value for next state

    #     q_values = self.training_network(states) # Q values from training network
    #     target_q_values = q_values.clone() # Create a copy to modify for targets

    #     for i in range(states.size(0)): # Iterate over batch
    #         action = actions[i]
    #         done = terminals[i]
    #         reward = rewards[i]

    #         if done:
    #             target_q_values[i, action] = reward
    #         else:
    #             target_q_values[i, action] = reward + self.gamma * max_next_q_values[i]

    #     return states, target_q_values

    def replay(self, states, actions, rewards, next_states, terminals):
        q_values_target = self.target_network(next_states).detach() # Detach from computation graph for target network
        max_next_q_values = q_values_target.max(dim=1)[0] # Get max Q-value for next state

        q_values = self.training_network(states) # Q values from training network
        target_q_values = q_values.clone() # Create a copy to modify for targets

        for i in range(states.size(0)): # Iterate over batch
            action = actions[i]
            done = terminals[i]
            reward = rewards[i]

            if done:
                target_q_values[i, action] = reward
            else:
                target_q_values[i, action] = reward + self.gamma * max_next_q_values[i]

        return states, target_q_values

    # def update_target_network(self):
    #     self.target_network.load_state_dict(self.training_network.state_dict()) # Copy weights

    def update_target_network(self, tau=0.005):
        for target_param, train_param in zip(self.target_network.parameters(), self.training_network.parameters()):
            target_param.data.copy_(tau * train_param.data + (1.0 - tau) * target_param.data)

    def save_model(self, path="../saved_models/ttt_dqn_second_final.pth"):
        torch.save(self.training_network.state_dict(), path)
        print(f"Model saved to {path}")


import wandb
wandb.login(key='e21de1f4d4c13b4ba109db92ba20cc946e7da3c5') 
wandb.init(project="experiments", name="ttt dqn going first")

env = Tictactoe_v0()
agent = DQN(0.95, 1, 4096, 100000)
agent.update_target_network()

num_episodes = 30001
batch_size = 32
epochs = 1
sample_size = 64
total_loss = 0
episode_count_for_avg_loss = 0
win_count = 0
loss_count = 0
draw_count = 0
training_steps = 0
target_update_freq = 1000

for episode in range(1, num_episodes + 1):
    state = env.reset(is_human_first=False)
    # print(f"[DEBUG] AI is playing as {'X' if env.player_mark == 1 else 'O'}")
    # print(f"[DEBUG] AI goes {'first' if env.player_mark == 1 else 'second'}")
    done = False
    episode_reward = 0

    while not done:
        action_space = [i for i, val in enumerate(state) if val == 0]
        action = agent.observe_on_training(state, action_space)
        next_state, reward, done, _ = env.step(action)
        agent.take_reward(reward, next_state, done)
        episode_reward = reward

        if agent.exp_replay.get_size() > agent.e_min:
            loss = agent.train_network(sample_size, batch_size, epochs, verbose=0)
            if loss is not None:
                total_loss += loss
                episode_count_for_avg_loss += 1
            training_steps += 1
            if training_steps % target_update_freq == 0:
                agent.update_target_network(tau=0.005)

        state = next_state

    agent.epsilon_greedy.decay(decay_value=0.995, lower_bound=0.01)

    if episode_reward == 1:
        win_count += 1
    elif episode_reward == -1:
        loss_count += 1
    else:
        if done:
            draw_count += 1

    if episode % 100 == 0:
        avg_loss = total_loss / episode_count_for_avg_loss if episode_count_for_avg_loss > 0 else 0
        print(f"Episode Summary {episode}, Games Won: {win_count}, Games Lost: {loss_count}, Games Drawn: {draw_count}")
        print(f"Episode: {episode}, Avg Loss: {avg_loss:.4f}")
        combined_rate = win_count + draw_count

        wandb.log({
            "Episode": episode,
            "Win Rate": win_count,
            "Loss Rate": loss_count,
            "Draw Rate": draw_count,
            "Win + Draw Rate": combined_rate,
            "Average Loss": avg_loss,
        })

        total_loss = 0
        episode_count_for_avg_loss = 0
        win_count = 0
        loss_count = 0
        draw_count = 0

agent.save_model()
wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/saideepa0501/.netrc
wandb: Currently logged in as: kradeero (kradeero-ohio-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Episode Summary 100, Games Won: 13, Games Lost: 81, Games Drawn: 6
Episode: 100, Avg Loss: 0.0000
Episode Summary 200, Games Won: 15, Games Lost: 73, Games Drawn: 12
Episode: 200, Avg Loss: 0.0000
Episode Summary 300, Games Won: 17, Games Lost: 78, Games Drawn: 5
Episode: 300, Avg Loss: 0.0000
Episode Summary 400, Games Won: 16, Games Lost: 75, Games Drawn: 9
Episode: 400, Avg Loss: 0.0000
Episode Summary 500, Games Won: 22, Games Lost: 66, Games Drawn: 12
Episode: 500, Avg Loss: 0.0000
Episode Summary 600, Games Won: 12, Games Lost: 81, Games Drawn: 7
Episode: 600, Avg Loss: 0.0000
Episode Summary 700, Games Won: 14, Games Lost: 79, Games Drawn: 7
Episode: 700, Avg Loss: 0.0000
Episode Summary 800, Games Won: 14, Games Lost: 78, Games Drawn: 8
Episode: 800, Avg Loss: 0.0000
Episode Summary 900, Games Won: 21, Games Lost: 71, Games Drawn: 8
Episode: 900, Avg Loss: 0.0000
Episode Summary 1000, Games Won: 20, Games Lost: 71, Games Drawn: 9
Episode: 1000, Avg Loss: 0.0000
Episode Summary 

Average Loss,▁▁█▆▆▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Draw Rate,▃▁▂▂▄▆▄▅▆▆▅▄▇▆▅▇▄▅▄▅▆▅▅▄▆▆▄▅▇▇▄▅▅█▄▆▆▆▇▆
Episode,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇█████
Loss Rate,█▇▇▇▄▄▂▂▂▃▂▂▂▁▁▂▂▂▁▂▂▁▂▂▁▂▂▂▂▂▂▂▂▂▁▂▂▃▂▂
Win + Draw Rate,▁▁▁▇█▇▇██▇▇███▇▇▇███▇█▇█▇▇██▇▇█▇▇█▇▇█▇█▇
Win Rate,▁▂███▇▇▇▆▇▇██▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▆▇█▇▆▆▇█▇▇▆▆
Average Loss,0.00014
Draw Rate,24
Episode,30000
Loss Rate,12
Win + Draw Rate,88
